In [ ]:
# !uv add langchain
# !uv add langchain-openrouter
# !uv add python-dotenv

Resolved 45 packages in 122ms                                        
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/22.72 KiB           
⠙ Preparing packages... (0/1)---------- 16.00 KiB/22.72 KiB         
⠙ Preparing packages... (0/1)---------- 22.72 KiB/22.72 KiB         
Prepared 1 package in 44ms                                                        
Installed 1 package in 2ms4                                 
 + python-dotenv==1.2.4


In [1]:
import requests
from dotenv import load_dotenv

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

from langchain_openrouter import ChatOpenRouter

load_dotenv()

True

In [2]:
# import os
# print(os.getenv("LANGSMITH_TRACING"), bool(os.getenv("LANGSMITH_API_KEY")))

In [3]:
@dataclass
class Context:
    user_id: str
    
@dataclass
class ResponseFormat:
    summary: str
    temperature_celsius: float
    temperature_fahrenheit: float
    humidity: float
    location: str

In [4]:
@tool('get_weather', description='Return weather information for a given city', return_direct=False)
def get_weather(city:str):
    response = requests.get(f'https://wttr.in/{city}?format=j1')
    return response.json()

@tool('locate_user', description="Look up a user's city based on context")
def locate_user(runtime: ToolRuntime[Context]):
    match runtime.context.user_id:
        case 'ABC123':
            return 'Delhi'
        case 'XYZ456':
            return 'London'
        case 'HJK789':
            return 'Paris'
        case _:
            return 'Unknown'

In [5]:
model = ChatOpenRouter(
    model="openrouter/free",
    temperature=0.2,
)

checkpointer = InMemorySaver()

In [6]:
agent = create_agent(
    model = model,
    tools = [get_weather, locate_user],
    system_prompt = 'You are a helpful weather assistant, who always cracks jokes and is humourous while remaining helpful.',
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

config = {
    'configurable' : {'thread_id' : 1}
}

response = agent.invoke({
    'messages':[
        {'role' : 'user', 'content' : 'What is the weather like?'}
    ]},
    config = config,
    context = Context(user_id='ABC123')
)

/Users/palak/PJ_Folders/Projects_Organiser/llm-engineering/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(


In [7]:
# print(response)
# print(response['messages'][-1].content)
print(response['structured_response'])

ResponseFormat(summary="The weather in Delhi is currently clear with a temperature of 25°C. It's a great day to be outside!", temperature_celsius=25.0, temperature_fahrenheit=77.0, humidity=45.0, location='Delhi')


In [8]:
print(response['structured_response'].summary)

The weather in Delhi is currently clear with a temperature of 25°C. It's a great day to be outside!


In [9]:
print(response['structured_response'].temperature_celsius)

25.0


In [10]:
print(response['structured_response'].location)

Delhi
